# PixelCNN — Conditional Image Generation with PixelCNN Decoders (2016)

_Papers · Generative Models_

**Generate an image one pixel at a time, left-to-right top-to-bottom, by making each pixel a classifier that may only look at the pixels already drawn — enforced with a masked convolution.**

---

This notebook is a practice scaffold for the **PixelCNN — Conditional Image Generation with PixelCNN Decoders (2016)** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)

# --- 0. Sanity-check the worked example: masked 3x3 conv at one pixel. -----------------
X = torch.tensor([[1., 2., 3.],
                  [4., 9., 6.],     # center pixel = 9 (the one we'd be predicting)
                  [7., 8., 5.]])
W = torch.ones(3, 3)               # all-ones filter, for clarity
def mask(kind):                    # 1 for past positions; center per A/B; 0 for future
    m = torch.zeros(3, 3)
    m[0, :] = 1                     # whole row above center -> past
    m[1, 0] = 1                     # center row, strictly left -> past
    if kind == "B":
        m[1, 1] = 1                 # mask B additionally allows the center
    return m                        # (kind == "A" leaves center = 0)
out_A = (X * W * mask("A")).sum().item()   # center (9) masked out
out_B = (X * W * mask("B")).sum().item()   # center allowed
out_full = (X * W).sum().item()            # unmasked: sees center + future
print("mask A:", out_A, " mask B:", out_B, " unmasked:", out_full)   # 10  19  45
assert out_A == 10 and out_B == 19 and out_full == 45
assert out_B - out_A == 9          # they differ by exactly the center pixel value

# --- 1. MaskedConv2d: zero the future weights BEFORE convolving (PixelRNN Sec 3.4). -----
class MaskedConv2d(nn.Conv2d):
    def __init__(self, mask_type, *args, **kwargs):
        super().__init__(*args, **kwargs)
        assert mask_type in ("A", "B")
        self.register_buffer("mask", torch.zeros_like(self.weight))
        _, _, kH, kW = self.weight.shape
        yc, xc = kH // 2, kW // 2
        self.mask[:, :, :yc, :] = 1            # all rows above center -> allowed
        self.mask[:, :, yc, :xc] = 1           # center row, strictly left -> allowed
        if mask_type == "B":
            self.mask[:, :, yc, xc] = 1         # mask B also allows the center
        # everything below / to the right stays 0 (the future)
    def forward(self, x):
        return F.conv2d(x, self.weight * self.mask, self.bias,
                        self.stride, self.padding, self.dilation, self.groups)

# --- 2. Tiny PixelCNN: mask-A input layer, mask-B hidden layers, 1x1 conv to a logit. ---
def make_pixelcnn(masked=True, C=32, k=7, n_hidden=4):
    pad = k // 2
    first = MaskedConv2d("A", 1, C, k, padding=pad) if masked \
            else nn.Conv2d(1, C, k, padding=pad)     # ablation: unmasked first layer
    layers = [first, nn.ReLU()]
    for _ in range(n_hidden):
        layers += [MaskedConv2d("B", C, C, k, padding=pad), nn.ReLU()]
    layers += [nn.Conv2d(C, 1, 1)]                   # 1x1 -> per-pixel logit (binary => sigmoid)
    return nn.Sequential(*layers)

# --- 3. Toy binary dataset: 8x8 images, each a single random horizontal OR vertical bar. -
def make_data(n, S=8):
    imgs = torch.zeros(n, 1, S, S)
    for i in range(n):
        idx = torch.randint(0, S, (1,)).item()
        if torch.rand(1).item() < 0.5: imgs[i, 0, idx, :] = 1.0   # horizontal bar
        else:                          imgs[i, 0, :, idx] = 1.0   # vertical bar
    return imgs
S = 8
train = make_data(512, S)

def train_model(masked, steps=400):
    torch.manual_seed(0)
    net = make_pixelcnn(masked=masked)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    losses = []
    for t in range(steps):
        x = train[torch.randint(0, len(train), (64,))]
        logits = net(x)                                   # one parallel pass: a logit per pixel
        loss = F.binary_cross_entropy_with_logits(logits, x)   # per-pixel cross-entropy
        opt.zero_grad(); loss.backward(); opt.step()
        losses.append(loss.item())
    return net, losses

net_masked, loss_m = train_model(True)
net_ablate, loss_a = train_model(False)   # first layer UNMASKED -> can cheat
print(f"final loss  masked {loss_m[-1]:.4f}   unmasked-ablation {loss_a[-1]:.4f}")
# The unmasked ablation's loss crashes far lower -- it copies each pixel through (leakage).

# --- 4. Generate pixel-by-pixel in raster order (the autoregressive sampling loop). ------
@torch.no_grad()
def sample(net, S=8):
    img = torch.zeros(1, 1, S, S)
    for r in range(S):
        for c in range(S):                                # strict raster order
            logit = net(img)[0, 0, r, c]                  # conditional for pixel (r,c)
            img[0, 0, r, c] = torch.bernoulli(torch.sigmoid(logit))
    return img[0, 0]

print("masked-model sample (a clean bar emerges):")
print(sample(net_masked, S).int().numpy())
print("ablated-model sample (incoherent -- leakage doesn't transfer to sampling):")
print(sample(net_ablate, S).int().numpy())
# Masked: pixel-by-pixel generation yields a coherent single bar.
# Ablation: low training loss but garbage samples -- masking is the load-bearing piece.
# (Our small CPU run on toy binary bars, not the paper's reported bits/dim.)

## Visualize the data & results

_Does removing the first-layer mask let the model 'cheat' — driving training loss far below the properly-masked model (the telltale sign of pixel leakage)?_

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

# Reproduces PixelCNN's qualitative point on toy data: removing the first-layer mask
# lets the model cheat (loss -> 0 via pixel leakage), so we log per-pixel loss for the
# masked model vs the unmasked-first-layer ablation, identical otherwise.
torch.manual_seed(0)

class MaskedConv2d(nn.Conv2d):
    def __init__(self, mask_type, *a, **k):
        super().__init__(*a, **k)
        self.register_buffer("mask", torch.zeros_like(self.weight))
        _, _, kH, kW = self.weight.shape; yc, xc = kH // 2, kW // 2
        self.mask[:, :, :yc, :] = 1
        self.mask[:, :, yc, :xc] = 1
        if mask_type == "B": self.mask[:, :, yc, xc] = 1
    def forward(self, x):
        return F.conv2d(x, self.weight * self.mask, self.bias,
                        self.stride, self.padding, self.dilation, self.groups)

def make_net(masked, C=32, k=7, nh=4):
    pad = k // 2
    first = MaskedConv2d("A", 1, C, k, padding=pad) if masked else nn.Conv2d(1, C, k, padding=pad)
    L = [first, nn.ReLU()]
    for _ in range(nh): L += [MaskedConv2d("B", C, C, k, padding=pad), nn.ReLU()]
    L += [nn.Conv2d(C, 1, 1)]
    return nn.Sequential(*L)

S = 8
def make_data(n):
    im = torch.zeros(n, 1, S, S)
    for i in range(n):
        j = torch.randint(0, S, (1,)).item()
        if torch.rand(1).item() < 0.5: im[i, 0, j, :] = 1.0
        else:                          im[i, 0, :, j] = 1.0
    return im
train = make_data(512)

def run(masked, steps=400):
    torch.manual_seed(0)
    net = make_net(masked); opt = torch.optim.Adam(net.parameters(), 1e-3); hist = []
    for t in range(steps):
        x = train[torch.randint(0, len(train), (64,))]
        loss = F.binary_cross_entropy_with_logits(net(x), x)
        opt.zero_grad(); loss.backward(); opt.step(); hist.append(loss.item())
    return hist

mask_hist = run(True); abl_hist = run(False)
idx = [0,10,20,30,40,50,70,90,110,140,170,200,240,280,320,360,399]
print("masked  :", [[i, round(mask_hist[i], 4)] for i in idx])
print("unmasked:", [[i, round(abl_hist[i], 4)] for i in idx])
# Masked loss settles ~0.14; unmasked crashes toward 0 (leakage). The unmasked model
# nonetheless GENERATES garbage -- low train loss does not imply a valid sampler.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation. You have a tiny PixelCNN whose first layer uses mask A; it trains to a
            moderate per-pixel loss and can generate recognizable binary shapes. Now remove the mask from
            the first layer (a plain conv that can see the center pixel), keep everything else identical, and
            retrain. What happens to (a) the training loss and (b) the quality of pixel-by-pixel samples, and
            which property of PixelCNN does this isolate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Change only the first layer: MaskedConv2d("A", ...) &rarr; an ordinary
                 nn.Conv2d with the same shape. Leave the mask-B layers, the loss, the optimizer, and
                 the data untouched. — _An honest ablation flips exactly one switch &mdash; whether the model
                 may see the pixel it predicts &mdash; so any change is attributable to masking._
- Retrain and watch the training loss: it plunges toward ~0, far below the masked model's
                 loss. — _Without mask A the network can copy the input pixel straight to the output, so
                 "predicting" $x_i$ becomes reading $x_i$ &mdash; a trivial, cheating solution with near-zero loss._
- Now generate pixel-by-pixel (sample $x_i$, write it back, continue). The samples are
                 incoherent noise, not shapes. — _At generation time the cheated dependency is
                 useless: the pixel it learned to copy isn't drawn yet, so every prediction rests on nothing
                 real. Low training loss does not transfer to sampling._
- Conclude that the mask is what makes the learned likelihood a valid autoregressive model,
                 not decoration. — _The factorization (Eq. 1) is only correct if each conditional depends on
                 past pixels alone; mask A is the mechanism that guarantees it._

**Answer:** Removing the first-layer mask lets the model cheat: training loss collapses toward $0$
                 (it just copies the target pixel through), but generation breaks &mdash; the samples are
                 incoherent because at sampling time the pixel it learned to read does not exist yet. This isolates
                 the masked convolution / mask A as the load-bearing piece: it is what makes Eq. 1's
                 factorization a valid likelihood rather than a self-referential fraud. The CODEVIZ panel shows the
                 masked model's loss settling at a sensible value while the unmasked model's loss crashes &mdash;
                 the telltale sign of leakage.

</details>

**Problem 2.** Hand-compute a masked convolution. For the patch $X=\begin{bmatrix}2&0&1\\3&5&4\\1&1&0\end{bmatrix}$
            and the all-ones $3\times3$ filter, what does a mask A convolution output at the center, and
            what does a mask B convolution output? Verify the difference is exactly the center pixel value.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Write mask A: top row all $1$, center-row-left $=1$, center $=0$, everything from the center
                 rightward and the whole bottom row $=0$:
                 $M_A=\begin{bmatrix}1&1&1\\1&0&0\\0&0&0\end{bmatrix}$. — _Mask A keeps only pixels strictly before the center in raster order, and forbids the center
                  itself._
- Mask-A sum $= (2{+}0{+}1) + (3) + 0 = 3 + 3 = 6$. — _Top row $2{+}0{+}1=3$; center row contributes only the left neighbor $3$; bottom row is future,
                  contributes $0$._
- Mask B is $M_A$ with center $=1$, so mask-B sum $= 6 + (5\cdot1) = 11$. — _Mask B additionally allows the center connection; the only new term is the center value $5$._
- Check: $11 - 6 = 5$, exactly the center pixel value. — _Mask A and mask B differ in precisely one weight &mdash; the center &mdash; so their outputs
                  differ by (center value $\times$ that weight) $= 5\times1 = 5$._

**Answer:** Mask A outputs $6$ (top row $3$ + left neighbor $3$ + nothing from the future). Mask B outputs
                 $11$, which is $6$ plus the center pixel value $5$. The difference $11-6=5$ is exactly the center
                 pixel &mdash; confirming the only distinction between mask A and mask B is whether a pixel
                 may see its own value. (This is why mask A goes on the input layer and mask B on the rest.)

</details>

**Problem 3.** The gated activation unit (Eq. 2) is $y=\tanh(W_f * x)\odot\sigma(W_g * x)$. Suppose at one feature
            position the content branch gives $\tanh(\cdot)=0.8$ and the gate branch gives $\sigma(\cdot)=0.1$.
            What is $y$ there, and what would $y$ be if the gate were instead $\sigma(\cdot)=0.95$? What is the
            gate doing?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compute the first case: $y = 0.8 \times 0.1 = 0.08$. — _The unit multiplies content by gate element-wise; a gate near $0$ nearly shuts the feature off._
- Compute the second case: $y = 0.8 \times 0.95 = 0.76$. — _A gate near $1$ lets almost the full content value pass._
- Note the content value $0.8$ was identical in both cases; only the gate changed the output. — _The sigmoid gate is a learned, input-dependent volume knob in $[0,1]$ on each content channel._

**Answer:** With gate $0.1$: $y=0.8\times0.1=0.08$ &mdash; the feature is almost suppressed. With gate
                 $0.95$: $y=0.8\times0.95=0.76$ &mdash; nearly the full content passes. The sigmoid branch is a
                 per-element gate (a learned $[0,1]$ multiplier) controlling how much of the tanh content
                 reaches the output. This multiplicative gating is more expressive than a single ReLU and is what
                 let Gated PixelCNN match the recurrent PixelRNN (&sect;2.1).

</details>